# Keyword Classification

This notebook shows the process of the keyword classifcation used within the study

In [ ]:
import pandas as pd
import multiprocessing as mp
import numpy as np
import regex as re

import nltk
from nltk.tokenize import sent_tokenize
from nltk.corpus import stopwords
from nltk.util import ngrams

from collections import Counter

nltk.download('stopwords')
nltk.download('punkt')

stopwords = set(stopwords.words('english'))

pd.set_option('display.max_rows', 50)
pd.set_option('display.max_colwidth', None)

df = pd.read_parquet("/Users/emma/Desktop/thesis/commentDataClean.parquet", engine="pyarrow")

In [69]:
df = df[~df['body'].isin(['removed', 'deleted'])]
dfs = df.sample(frac=.05)

In [70]:
dfs

,body,author,score,id,link_id,subreddit,author_flair_text,time
2357311,heres the thing his justices wont pick and choose like this. the same justices that are against abortion are also the same justices against same sex marriage.,PARK_THE_BUS,141,d9z5l6u,t3_5ct4ho,politics,None,2016-11-14 00:38:46
775937,i cant wait for you all to lose in the midterms. commies gonna commie and it is gonna be so good to watch you lose again.,mongoose-american,0,i8inz2a,t3_uotfr7,politics,None,2022-05-14 00:06:30
2209605,but he does represent the thinking of a large portion of the people in his district at least the ones who vote for him thats the sad part of the election of some of these tea party people. theres a huge disconnect in this country with a lot of rural people especially with a lot of backwards and religious thinking versus the urban and suburban people. thats really at the heart of this problem and these tea partiers being elected is just a big symptom.,freediverdude,4,c6otwrx,t3_11qs41,politics,None,2012-10-19 15:13:13
1262972,the reality is this story is probably going to get drowned out tomorrow when the supreme court announces its samesex marriage decision.,stilgar02,22,caqev15,t3_1h1lfy,politics,None,2013-06-26 05:37:04
1119667,its far more important to protect the public from hearing protest chants than to protect children from being rapedtraffickedetc.. s,cuisinart-hatrack,2446,iguiqrg,t3_w2zsxr,politics,None,2022-07-19 23:33:47
...,...,...,...,...,...,...,...,...
986949,not fee fees so much as human lives lost. the discussion is an important one even if its not threat or priority 1. but since you apparently prioritize lightning deaths as the statistical likelihood comparatively how shall we tackle that first,that_one_bastard,0,cxo8yij,t3_3vjaqm,politics,None,2015-12-05 18:39:45
1046007,no. he would leave it to the state. also if a doctor kills a fetus in the 7th month of pregnancy on purpose isnt he liable for murder,alragusa,1,c2fs8t7,t3_jwo4d,politics,None,2011-08-28 13:12:05
69518,i never claimed alive meant person though or maybe i miswrote something,calamityfriends,-6,igbov16,t3_vznddd,politics,None,2022-07-15 22:45:44
2503821,democrats are baby killers just like the phrase they yelled at vietnam veterans.,ShoutingMatch,8,g4hjebn,t3_ip3ake,conservative,,2020-09-08 22:12:39


In [ ]:
pck = [
    "my choice", "bodily autonomy", "body autonomy", "womens rights", "reproductive rights",
    "abortion rights", "forced birth", "government control", "privacy", "unsafe abortions", 'christofacists', 
    "anti choice", "im prochoice", "fetus"
]

plk = [
    "abortion is murder", "unborn child", "killing babies", 'sanctity of life', "im prolife",
    "sanctity of life", "right to life", "protect the unborn", 
    "anti life", 'innocent', 'life begins at conception',
    "life liberty and the pursuit", "killing"
]


dfs['test'] = ""

def sent_score(text, pck, plk):
    pc_score = sum(1 for word in pck if word in text.lower())
    pl_score = sum(1 for word in plk if word in text.lower())

    if pc_score == 0 and pl_score == 0:
        return "None"
    if pc_score > pl_score:
        return "prochoice"
    if pc_score == pl_score:
        return "unclear"
    
    return "prolife"
    

dfs['test'] = dfs['body'].apply(lambda x: sent_score(x, pck, plk))


In [72]:
dfs['test'].value_counts()

test
None         113862
prochoice      2601
prolife        1658
unclear         115
Name: count, dtype: int64

In [75]:
pl = dfs[dfs['test'] == 'prolife']
pl[['body', 'test']].sample(10)

,body,test
1476536,thats not what i mean at all. the violation of the right to life was for everyone who used the system and had to wait so long. they introduced the private sector so if someone dies waiting they could say that if they wanted better treatment they could have gone to a private clinic so it was their choice to go to the public hospital. the problem is you will then be paying for both public and private care because you are still being taxed a crazy amount for public care even though you dont want to use it. you then loose the choice to go to the private clinic because you can no longer afford it because you tax rate is something like 30. right now only very wealthy people can afford private care in canada because they are the only ones who can afford paying 40 tax and paying for private treatment. so you end up with the middle class basically forced in to the public system and they end up getting the worse treatment.,prolife
875040,first with doctors you are simply denying care. with abortion you are actively scrambling the brains of someone else. thats why we have the triage system. it takes away all moral guesswork and is encoded into medical law. this is a far rarer situation but if conjoined twins came to the emergency room and one was dying youd separate them so the other can live. its exactly the same for a doomed embryo or fetus. if scrambling brains is wrong induce labor and withhold care. is that okay this is a part of the er job description. i dont think anyone should be forced to perform abortions if they dont want so take a different job that doesnt require it. now that doesnt mean you have to allow them to be on your property in the first place but once they are in that position they dont simply lose all their rights. the same people who fight to ban 100 of abortion support things like the castle doctrine. that doctrine got its start in america when a louisiana homeowner shot an innocent japanese exchange student who got the wrong house when trying to find a halloween party. the personal and property rights that affect men are sanctified by the rightwing. the gop is all for murder under just about every circumstance including executing the known innocent see cameron todd willingham and endless war. this sort of thing just makes you look like a crazy person it would make me look crazy if i didnt have hundreds of womankilling bills backing me up.,prolife
1161054,i just listened to the audio wooo bolding he is not saying they are similar. he is saying the choice about what to do with the baby is similar because the out of wedlock baby was unwanted and i assume the pregnancy itself shameful to him and that a choice had to be made by his daughter maybe whether to abort it or not. he really does not answer the question posed. how do you explain a no exceptions policy to a daughter or granddaughter that was raped and is now pregnant that they need to keep the baby against their will paraphrase for clarity he draws his analogy then just says im prolife period. you guys are looking for ill intent here and i dont think there is any. i agree this guys views on abortion are incorrect but i dont see where his comments make him a bad person deserving of scorn.,prolife
1838760,a bill that could allow the death penalty for women who receive abortions emerged as a pivotal issue in a state legislative race in texas. the legislation which was filed last year by state representative bryan slaton would allow women who receive an abortion to be charged with assault or homicide which carries the states death sentence. the battle over how republicans should handle the issue of abortion has become central in one gop primary in texas 91st congressional district which contains conservative suburbs north of fort worth. in the primary held earlier this month state representative stephanie klick was forced into a runoff with challenger david lowe. i support representative slatons bill lowe said. which was probably the strongest 

**NLTK Counter**

In [33]:
dfs['test'].value_counts()

test
None         11410
prochoice      248
prolife        156
unclear         10
Name: count, dtype: int64

In [ ]:
stance_df = dfs[dfs['test'].isin(['prochoice', 'prolife'])]

def tokenize(text, n):

    text= re.sub(r'[^\w\s]', '', text.lower())
    words = text.split()
    words = [word for word in words if word not in stopwords]
    return list(ngrams(words, n))

In [92]:
pro_choice = Counter()
pro_life = Counter()

for _, row in stance_df.iterrows():
    words = tokenize(row['body'], 3)
    if row['test'] == 'prochoice':
        pro_choice.update(words)
    elif row['test'] == 'prolife':
        pro_life.update(words)

pc_grams = set([bg for bg, _ in pro_choice.most_common(20)])
pl_grams = set([bg for bg, _ in pro_life.most_common(20)])

print('unique pc:', pc_grams - pl_grams)
print('unique pl:', pl_grams - pc_grams)



unique pc: {('whole', 'womans', 'health'), ('media', 'domain', 'privacy'), ('submission', 'automatically', 'removed'), ('roe', 'vs', 'wade'), ('pro', 'forced', 'birth'), ('automatically', 'removed', 'comes'), ('v', 'wade', 'court'), ('concerns', 'well', 'concerns'), ('support', 'womens', 'rights'), ('womens', 'reproductive', 'rights'), ('rights', 'womens', 'rights'), ('domain', 'privacy', 'concerns'), ('roe', 'v', 'wade'), ('comes', 'social', 'media'), ('privacy', 'concerns', 'well'), ('abortion', 'womens', 'rights'), ('rights', 'bodily', 'autonomy'), ('social', 'media', 'domain'), ('removed', 'comes', 'social')}
unique pl: {('believe', 'life', 'begins'), ('homeless', 'people', 'die'), ('one', 'right', 'murder'), ('think', 'abortion', 'murder'), ('people', 'like', 'trying'), ('abortion', 'murder', 'would'), ('numbers', 'chapter', '5'), ('life', 'liberty', 'pursuit'), ('killing', 'unborn', 'child'), ('right', 'life', 'liberty'), ('body', 'without', 'consent'), ('sexually', 'active', 'wo